# SARIMA (classical baseline)

SARIMA (Seasonal ARIMA) is a classical statistical model. It is per-series and, with a
yearly season (period 52 on weekly data), it is slow and numerically unstable on short
series. Following the guidance not to spend much time on classical models, we fit it on
a small sample and report the WMAE on those series' real validation rows.

Note: the raw seasonal forecasts can diverge, so predictions are clipped to a sane
range (0 to 2x the series' historical max). Logged to the shared DagsHub; reference
point only.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import yaml
import warnings
import numpy as np
import pandas as pd
import mlflow
import dagshub

from src.features.nn_preprocessing import build_long_df, temporal_split, get_real_validation, FREQ
from src.models.sarima_pipeline import build_sarima
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings("ignore")

# Shared DagsHub tracking.
dagshub.init(repo_owner="slosa23", repo_name="Walmart-Sales-Forecasting", mlflow=True)
mlflow.set_experiment("SARIMA_Training")

# Load the model config.
with open("configs/sarima_config.yaml") as f:
    config = yaml.safe_load(f)
params = config["model"]["params"]
split_date = config["data"]["split_date"]
SEED = 42
print("Tracking URI:", mlflow.get_tracking_uri())

D:\Desktop\ML Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

Tracking URI: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow


In [3]:
train_raw = pd.read_csv("data/train.csv")
stores = pd.read_csv("data/stores.csv")
features = pd.read_csv("data/features.csv")
print("train:", train_raw.shape)

train: (421570, 5)


## Run 1: Data Preparation

In [4]:
with mlflow.start_run(run_name="SARIMA_Data_Prep"):
    Y_df = build_long_df(train_raw, stores, features, fill_gaps=True)
    train_df, valid_df, horizon = temporal_split(Y_df, split_date)
    real_valid = get_real_validation(train_raw, stores, features, split_date)

    mlflow.log_param("freq", FREQ)
    mlflow.log_param("split_date", split_date)
    mlflow.log_param("sample_size", params["sample_size"])
    mlflow.log_param("order", str(params["order"]))
    mlflow.log_param("seasonal_order", str(params["seasonal_order"]))
    mlflow.log_param("horizon_weeks", int(horizon))

    print("series:", Y_df["unique_id"].nunique(), "| horizon:", horizon)

series: 3331 | horizon: 43
🏃 View run SARIMA_Data_Prep at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/9/runs/614777448c2b4a5a95bdbdea58cbb4a8
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/9


## Run 2: SARIMA on a sample of series

Fit one SARIMA per sampled series, forecast the validation horizon, clip to a sane
range, and score against each series' real validation rows. Fits that fail are skipped.

In [5]:
# Pick eligible series: long history (near-full) and present in the real validation.
train_counts = train_df.groupby("unique_id").size()
valid_ids = set(real_valid["unique_id"].unique())
eligible = [uid for uid in train_counts.index if train_counts[uid] >= 90 and uid in valid_ids]

rng = np.random.default_rng(SEED)
sample_ids = list(rng.choice(eligible, size=min(params["sample_size"], len(eligible)), replace=False))
print("eligible:", len(eligible), "| sampled:", len(sample_ids))

eligible: 3055 | sampled: 10


In [6]:
with mlflow.start_run(run_name="SARIMA_Subset"):
    mlflow.log_params({k: str(v) for k, v in params.items()})

    all_true, all_pred, all_hol = [], [], []
    fitted_count = 0
    for i, uid in enumerate(sample_ids, 1):
        tr = train_df[train_df["unique_id"] == uid].sort_values("ds")
        try:
            fitted = build_sarima(tr["y"].values, params)
            fc = np.asarray(fitted.forecast(steps=horizon))
        except Exception as e:
            print(f"  skipped {uid}: {type(e).__name__}")
            continue

        # Clip forecasts: sales are non-negative and won't explode past ~2x history.
        cap = 2.0 * float(tr["y"].max())
        ds_future = pd.date_range(tr["ds"].max() + pd.Timedelta(weeks=1), periods=horizon, freq=FREQ)
        fdf = pd.DataFrame({"ds": ds_future, "yhat": np.clip(fc, 0, cap)})
        va = real_valid[real_valid["unique_id"] == uid][["ds", "y", "IsHoliday"]]
        merged = va.merge(fdf, on="ds", how="inner")

        all_true.extend(merged["y"].tolist())
        all_pred.extend(merged["yhat"].tolist())
        all_hol.extend(merged["IsHoliday"].tolist())
        fitted_count += 1
        print(f"  fitted {i}/{len(sample_ids)}")

    wmae = calculate_wmae(np.array(all_true), np.array(all_pred), np.array(all_hol))
    mlflow.log_metric("WMAE", float(wmae))
    mlflow.log_param("fitted_series", fitted_count)
    print(f"SARIMA subset WMAE ({fitted_count} series, real validation): {wmae:,.2f}")

  fitted 1/10
  fitted 2/10
  fitted 3/10
  fitted 4/10
  fitted 5/10
  fitted 6/10
  fitted 7/10
  fitted 8/10
  fitted 9/10
  fitted 10/10
SARIMA subset WMAE (10 series, real validation): 2,344.50
🏃 View run SARIMA_Subset at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/9/runs/0198e80a94c7458abe71bc7128f0d38c
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/9


## Notes

- WMAE is on the real validation rows of a small sample, so it is a rough reference.
- Raw seasonal SARIMA forecasts diverged on short weekly series, so we clipped them;
  this instability is itself a reason global neural models are preferred here.
- Params live in `configs/sarima_config.yaml`; the model is built in
  `src/models/sarima_pipeline.py`.